# Dataset gate — nuScenes keyframes (ticket 04)

Mục tiêu: xác minh dataset Kaggle `sahangunasekara92/nuscenes-v1-0-full-keyframes`
có đủ cấu trúc để tạo infos và load một sample.

Cách dùng: attach dataset vào notebook, rồi chạy **từng ô một**. Mỗi ô chỉ
phụ thuộc vào các biến của ô trước, nên khi một ô lỗi chỉ cần sửa và chạy lại
ô đó. Không cần chạy lại từ đầu.

Ghi chú môi trường: Kaggle hiện dùng Python 3.13; `nuscenes-devkit==1.1.11`
pin `Shapely<2` nên pip build from source và fail. Ô devkit là **tùy chọn**
và đã có cách lách; các ô gate chính không cần devkit.

In [ ]:
import glob, hashlib, json, os, subprocess, sys

import numpy as np

VERSION = "v1.0-trainval"
CANDIDATES = [
    "/kaggle/input/datasets/sahangunasekara92/nuscenes-v1-0-full-keyframes",
    "/kaggle/input/nuscenes-v1-0-full-keyframes",
    *sorted(glob.glob("/kaggle/input/*")),
]
ROOT = os.environ.get("NUSCENES_ROOT") or next(
    (p for p in CANDIDATES if os.path.isdir(os.path.join(p, VERSION))),
    None,
)
assert ROOT, "dataset chua attach: them nuscenes-v1-0-full-keyframes"
print("ROOT:", ROOT)
print("top-level:", sorted(os.listdir(ROOT)))

In [ ]:
mdir = os.path.join(ROOT, VERSION)
names = [
    "scene", "sample", "sample_data", "sample_annotation", "instance",
    "category", "attribute", "visibility", "calibrated_sensor",
    "ego_pose", "log", "map",
]
metadata = {n: os.path.isfile(os.path.join(mdir, f"{n}.json")) for n in names}
print(metadata)
assert all(metadata.values()), "thieu metadata"

In [ ]:
def count_files(directory):
    with os.scandir(directory) as entries:
        return sum(entry.is_file() for entry in entries)


samples_dir = os.path.join(ROOT, "samples")
sensors = {
    name: count_files(os.path.join(samples_dir, name))
    for name in sorted(os.listdir(samples_dir))
}
print("sensors:", sensors)

sweeps_dir = os.path.join(ROOT, "sweeps")
has_sweeps = os.path.isdir(sweeps_dir)
print("sweeps present:", has_sweeps)
if has_sweeps:
    print({n: count_files(os.path.join(sweeps_dir, n)) for n in sorted(os.listdir(sweeps_dir))})

In [ ]:
license_path = os.path.join(ROOT, "LICENSE")
license_bytes = open(license_path, "rb").read()
license_sha256 = hashlib.sha256(license_bytes).hexdigest()
print("sha256:", license_sha256)
print("\n".join(license_bytes.decode("utf-8", errors="replace").strip().splitlines()[:4]))
print("maps:", count_files(os.path.join(ROOT, "maps")))

In [ ]:
samples = json.load(open(os.path.join(mdir, "sample.json")))
sample_data = json.load(open(os.path.join(mdir, "sample_data.json")))
by_token = {record["token"]: record for record in sample_data}
print("samples:", len(samples), "sample_data:", len(sample_data))

sample = samples[0]
print("sample:", sample["token"], "annotations:", len(sample["anns"]))

lidar = by_token[sample["data"]["LIDAR_TOP"]]
print("lidar key frame:", lidar["is_key_frame"], lidar["filename"])
lidar_path = os.path.join(ROOT, lidar["filename"])
points = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 5)
print("lidar points:", points.shape, "intensity:", points[:, 3].min(), points[:, 3].max())

channels = [
    "CAM_FRONT", "CAM_FRONT_RIGHT", "CAM_FRONT_LEFT",
    "CAM_BACK", "CAM_BACK_LEFT", "CAM_BACK_RIGHT",
]
cams = {}
for channel in channels:
    record = by_token[sample["data"][channel]]
    path = os.path.join(ROOT, record["filename"])
    cams[channel] = {
        "exists": os.path.isfile(path),
        "bytes": os.path.getsize(path) if os.path.isfile(path) else 0,
    }
print(cams)
assert all(cam["exists"] for cam in cams.values())

In [ ]:
# TUY CHON: nuscenes-devkit. Neu o nay loi, cac o gate phia tren van dung.
# Kaggle Python 3.13 + devkit 1.1.11 pin Shapely<2 -> cai Shapely 2 truoc,
# roi cai devkit khong keo dependency.
def run(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=False)

run(
    f"{sys.executable} -m pip install -q 'shapely==2.0.7' pyquaternion "
    "descartes scikit-learn tqdm opencv-python-headless matplotlib Pillow"
)
run(f"{sys.executable} -m pip install -q nuscenes-devkit==1.1.11 --no-deps")

try:
    from nuscenes.nuscenes import NuScenes

    nusc = NuScenes(version=VERSION, dataroot=ROOT, verbose=False)
    print("devkit scenes:", len(nusc.scene), "samples:", len(nusc.sample))
except Exception as exc:
    print("devkit unusable in this image:", repr(exc))

In [ ]:
summary = {
    "root": ROOT,
    "version": VERSION,
    "scenes": len(json.load(open(os.path.join(mdir, "scene.json")))),
    "samples": len(samples),
    "annotations": len(json.load(open(os.path.join(mdir, "sample_annotation.json")))),
    "metadata": metadata,
    "sensors": sensors,
    "sweeps_present": has_sweeps,
    "maps": count_files(os.path.join(ROOT, "maps")),
    "license_sha256": license_sha256,
    "sample_check": {
        "token": sample["token"],
        "lidar_shape": list(points.shape),
        "cams": cams,
        "annotations": len(sample["anns"]),
    },
}
print(json.dumps(summary, indent=2))
out_dir = os.environ.get("NUSCENES_OUT", "/kaggle/working")
os.makedirs(out_dir, exist_ok=True)
json.dump(summary, open(os.path.join(out_dir, "dataset_gate.json"), "w"), indent=2)

## Ghi lại kết quả

Dán `dataset_gate.json` vào ticket 04. Cần trả lời:

- danh sách sensor và số file mỗi loại;
- có `sweeps/` hay không (ảnh hưởng sweep policy của ticket 10/11);
- một keyframe load được: shape point cloud, 6 camera tồn tại;
- license và sha256;
- phần thiếu (nếu có).

Lưu ý: dataset này không có `sweeps/`, nên training phải dùng 0 sweep.
Việc tạo infos bằng converter MIT phụ thuộc toolchain ở ticket 02 và chưa
chạy trong notebook này.